# Bronze — Landing → Delta (incremental, tipos preservados)

Issue #38. Lê os CSVs da Landing, aplica os **tipos corretos** (a origem é
relacional, então o schema é conhecido), adiciona metadados de ingestão e
persiste em **Delta Lake**. A carga é **incremental e idempotente** via
`MERGE` pela chave primária — reexecutar não duplica.

Padrão de path: `s3a://<bucket>/bronze/olist/<tabela>` (tabela Delta).

In [ ]:
import os

run_date = "2026-06-17"

minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_secret_key = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.environ.get("DATALAKE_BUCKET", "datalake")

In [ ]:
APP_NAME = "engdb-bronze-olist"
EXTRA_PACKAGES = ["org.apache.hadoop:hadoop-aws:3.3.4"]

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName(APP_NAME)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# Metadados de cada tabela: chave primária e casts (colunas não citadas ficam string).
# ts = timestamp; demais usam cast direto do Spark.
TABLES = {
    "categories":  {"pk": ["category_id"],  "casts": {"category_id": "int"}},
    "customers":   {"pk": ["customer_id"],  "casts": {"customer_zip_code_prefix": "int"}},
    "sellers":     {"pk": ["seller_id"],    "casts": {"seller_zip_code_prefix": "int"}},
    "products":    {"pk": ["product_id"],   "casts": {
        "category_id": "int", "product_name_length": "int", "product_description_length": "int",
        "product_photos_qty": "int", "product_weight_g": "int", "product_length_cm": "int",
        "product_height_cm": "int", "product_width_cm": "int"}},
    "orders":      {"pk": ["order_id"],     "casts": {
        "order_purchase_timestamp": "ts", "order_approved_at": "ts",
        "order_delivered_carrier_date": "ts", "order_delivered_customer_date": "ts",
        "order_estimated_delivery_date": "ts"}},
    "addresses":   {"pk": ["address_id"],   "casts": {}},
    "payments":    {"pk": ["payment_id"],   "casts": {
        "payment_sequential": "int", "payment_installments": "int", "payment_value": "decimal(12,2)"}},
    "reviews":     {"pk": ["review_id"],    "casts": {
        "review_score": "int", "review_creation_date": "ts", "review_answer_timestamp": "ts"}},
    "shipments":   {"pk": ["shipment_id"],  "casts": {"shipped_at": "ts", "delivered_at": "ts"}},
    "order_items": {"pk": ["order_item_id"],"casts": {
        "order_item_id": "int", "shipping_limit_date": "ts",
        "price": "decimal(12,2)", "freight_value": "decimal(12,2)"}},
}

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

TS_FMT = "yyyy-MM-dd HH:mm:ss"

for t, meta in TABLES.items():
    src = f"s3a://{bucket}/landing/olist/{t}/ingestion_date={run_date}"
    df = spark.read.option("header", "true").option("nullValue", "").csv(src)

    # tipagem explícita (tipos preservados a partir da origem relacional)
    for col, typ in meta["casts"].items():
        if col in df.columns:
            if typ == "ts":
                df = df.withColumn(col, F.to_timestamp(F.col(col), TS_FMT))
            else:
                df = df.withColumn(col, F.col(col).cast(typ))

    # metadados de ingestão (linhagem)
    df = (df.withColumn("_ingestion_date", F.lit(run_date))
            .withColumn("_bronze_loaded_at", F.current_timestamp()))

    path = f"s3a://{bucket}/bronze/olist/{t}"
    if DeltaTable.isDeltaTable(spark, path):
        cond = " AND ".join([f"t.{k} = s.{k}" for k in meta["pk"]])
        dt = DeltaTable.forPath(spark, path)
        (dt.alias("t").merge(df.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "merge"
    else:
        df.write.format("delta").mode("overwrite").save(path)
        mode = "init "
    total = spark.read.format("delta").load(path).count()
    print(f"[bronze] {t:13} {mode} linhas={total:>7}")

print("Bronze concluída.")
spark.stop()